# Brain Tumour Classification from MRI — CNN (Kaggle GPU)

Four-way classification of brain MRI slices: **glioma**, **meningioma**,
**pituitary tumour**, and **no tumour**.

### Setup on Kaggle
1. **Add Input** -> search `brain-tumor-classification-mri` -> add
   `sartajbhuvaji/brain-tumor-classification-mri`
2. **Settings -> Accelerator -> GPU T4 x2**
3. **Run All** (roughly 6-10 minutes)

### Approach
Transfer learning from ImageNet with MobileNetV2. The backbone is frozen first
so the randomly-initialised head can settle without destroying the pretrained
filters, then the top of the backbone is unfrozen and fine-tuned at a much lower
learning rate. With only ~2,870 training images this beats training from
scratch — ImageNet contains no MRIs, but the early edge and texture filters
transfer regardless.

Outputs `brain_tumor.keras`, which drops straight into the repo's
`models/image/` folder.

In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "NONE - set Settings > Accelerator > GPU T4 x2")

In [ ]:
BASE = "/kaggle/input/brain-tumor-classification-mri"
IMG_SIZE = 224
BATCH = 32
CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
PRETTY = ["Glioma", "Meningioma", "No tumour", "Pituitary"]

for split in ("Training", "Testing"):
    print(split)
    for c in CLASSES:
        p = os.path.join(BASE, split, c)
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"   {c:<20} {n:>5}")

## 1. Datasets

The 20% validation split is carved out of *Training*. The supplied *Testing*
folder is touched only once, at the very end — using it to pick the best epoch
would quietly turn it into a validation set and inflate the final number.

In [ ]:
common = dict(image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
              label_mode="int", class_names=CLASSES)

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/Training", validation_split=0.2, subset="training", seed=SEED, **common)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/Training", validation_split=0.2, subset="validation", seed=SEED, **common)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/Testing", shuffle=False, **common)

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(13, 7))
for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(PRETTY[int(labels[i])], fontsize=11)
        plt.axis("off")
plt.suptitle("Brain MRI training samples", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()

## 2. Model

In [ ]:
def build_model(img_size=IMG_SIZE):
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.06),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ], name="augment")

    base = tf.keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = augment(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs), base

model, base = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 3. Stage 1 — frozen backbone

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=3, min_lr=1e-6, verbose=1),
]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

## 4. Stage 2 — fine-tuning

The last 40 layers of the backbone are unfrozen and trained at 1e-5. A higher
rate here would wash out the pretrained weights — the whole reason for using
them.

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("trainable weights:", len(model.trainable_weights))
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
def plot_history(histories, labels):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    off = 0
    for h, lab in zip(histories, labels):
        e = range(off, off + len(h.history["accuracy"]))
        ax[0].plot(e, h.history["accuracy"], label=f"{lab} train")
        ax[0].plot(e, h.history["val_accuracy"], "--", label=f"{lab} val")
        ax[1].plot(e, h.history["loss"], label=f"{lab} train")
        ax[1].plot(e, h.history["val_loss"], "--", label=f"{lab} val")
        off += len(h.history["accuracy"])
    ax[0].set_title("Accuracy"); ax[1].set_title("Loss")
    for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=8); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_history([hist1, hist2], ["frozen", "fine-tune"])

## 5. Evaluate on the held-out Testing folder

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}   loss: {test_loss:.4f}\n")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=PRETTY, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=PRETTY,
            yticklabels=PRETTY, cbar=False, ax=ax[0])
ax[0].set_title("Counts"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".1%", cmap="Blues",
            xticklabels=PRETTY, yticklabels=PRETTY, cbar=False, ax=ax[1])
ax[1].set_title("Row-normalised"); ax[1].set_xlabel("Predicted")
plt.suptitle(f"Brain tumour CNN - test accuracy {test_acc:.2%}", weight="bold")
plt.tight_layout(); plt.show()

### Which mistakes matter

Not all errors are equal here. Predicting **no tumour** for a scan that has one
is far worse than confusing two tumour types, since the second still sends the
patient for review. The cell below reports that miss rate separately.

In [ ]:
NO_TUMOUR = CLASSES.index("no_tumor")
has_tumour = y_true != NO_TUMOUR
missed = (y_pred[has_tumour] == NO_TUMOUR).sum()
print(f"Scans with a tumour: {has_tumour.sum()}")
print(f"Called 'no tumour' anyway (false negatives): {missed} "
      f"({missed / has_tumour.sum():.2%})")

healthy = ~has_tumour
false_alarm = (y_pred[healthy] != NO_TUMOUR).sum()
print(f"\nHealthy scans: {healthy.sum()}")
print(f"Flagged as a tumour (false positives): {false_alarm} "
      f"({false_alarm / max(healthy.sum(), 1):.2%})")

## 6. Save

In [ ]:
model.save("/kaggle/working/brain_tumor.keras")
metrics = {
    "task": "brain_tumor",
    "classes": PRETTY,
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "confusion_matrix": cm.tolist(),
    "tumour_false_negative_rate": float(missed / has_tumour.sum()),
    "classification_report": classification_report(
        y_true, y_pred, target_names=PRETTY, output_dict=True),
}
with open("/kaggle/working/brain_tumor_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("saved:", os.listdir("/kaggle/working"))

---

Download `brain_tumor.keras` from the **Output** panel into the repo's
`models/image/` folder, and `brain_tumor_metrics.json` into `reports/`.
`streamlit run app.py` will then pick it up under **Medical imaging (CNN)**.